In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install evaluate rouge_score -q
!pip install transformers==4.40.0 -q


Mounted at /content/drive
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 102.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 137.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BartTokenizer
import torch

MODEL_PATH   = "/content/drive/MyDrive/final_model"
PROJECT_PATH = "/content/drive/MyDrive/Project_1"  

print("Loading tokenizer from facebook/bart-base...")
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
print("✅ Tokenizer loaded")

print("Loading your fine-tuned model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model.eval()
print("✅ Model loaded")

def summarise(text, max_length=60, min_length=5):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )
    with torch.no_grad():
        output = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

test_df = pd.read_csv(f"{PROJECT_PATH}/test.csv")
print(f"✅ test.csv loaded — {len(test_df)} rows")

predictions, references = [], []
print("\n⏳ Running predictions on test set...")

for i, (_, row) in enumerate(test_df.iterrows()):
    pred = summarise(row["findings"])
    predictions.append(pred)
    references.append(row["impression"])

    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(test_df)} rows...")

rouge  = evaluate.load("rouge")
scores = rouge.compute(predictions=predictions, references=references)

print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"✅ ROUGE1 F1 Score: {round(scores['rouge1'], 4)}")
print(f"   ROUGE2 F1 Score: {round(scores['rouge2'], 4)}")
print(f"   ROUGEL F1 Score: {round(scores['rougeL'], 4)}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

if scores["rouge1"] >= 0.3:
    print("🎉 PASSED — Score meets the minimum 0.3 requirement!")
else:
    print("⚠️  FAILED — Score below 0.3, consider more training epochs")

print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("📋 Sample Predictions from Spec Appendix")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")

sample_findings = [
    "The trachea is midline. The cardiomediastinal silhouette is normal. The lungs are clear, without evidence of acute infiltrate or effusion. There is no pneumothorax. The visualized bony structures reveal no acute abnormalities.",
    "The lungs are clear. Heart size and mediastinal contours are normal. No osseous abnormalities.",
    "AP and lateral views were obtained. Bibasilar atelectasis and small left-sided pleural effusion. Stable cardiomegaly. No pneumothorax. Mild pulmonary vascular congestion."
]

for i, finding in enumerate(sample_findings):
    impression = summarise(finding)
    print(f"Sample {i+1}:")
    print(f"  FINDINGS:   {finding}")
    print(f"  IMPRESSION: {impression}")
    print()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading tokenizer from facebook/bart-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded
Loading your fine-tuned model...
✅ Model loaded
✅ test.csv loaded — 684 rows

⏳ Running predictions on test set...


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1256: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


  Processed 50/684 rows...
  Processed 100/684 rows...
  Processed 150/684 rows...
  Processed 200/684 rows...
  Processed 250/684 rows...
  Processed 300/684 rows...
  Processed 350/684 rows...
  Processed 400/684 rows...
  Processed 450/684 rows...
  Processed 500/684 rows...
  Processed 550/684 rows...
  Processed 600/684 rows...
  Processed 650/684 rows...



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ ROUGE1 F1 Score: 0.6006
   ROUGE2 F1 Score: 0.5117
   ROUGEL F1 Score: 0.5953
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎉 PASSED — Score meets the minimum 0.3 requirement!

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 Sample Predictions from Spec Appendix
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Sample 1:
  FINDINGS:   The trachea is midline. The cardiomediastinal silhouette is normal. The lungs are clear, without evidence of acute infiltrate or effusion. There is no pneumothorax. The visualized bony structures reveal no acute abnormalities.
  IMPRESSION: No acute cardiopulmonary abnormalities..

Sample 2:
  FINDINGS:   The lungs are clear. Heart size and mediastinal contours are normal. No osseous abnormalities.
  IMPRESSION: Clear lungs.

Sample 3:
  FINDINGS:   AP and lateral views were obtained. Bibasilar atelectasis and small left-sided pleural effusion. Stable cardiomegaly. No pneumothorax. Mild pulmonary vascular congestion.
  IMPRESSION: 1. No acute cardio